In [2]:
import requests

# My API key
api_key = "fp_b1240f0a75d60c45b30f200372d5b9206b866b053de347d168dc9e00abd438da"

# API endpoint
url = "https://fireping.net/api/v1/locations"  

# Headers for authentication
headers = {
    "Authorization": f"Bearer {api_key}"
}

# Make the GET request
response = requests.get(url, headers=headers)

print(response.status_code)
print(response.json())


200
{'data': [{'id': '6c8504da-0453-4c7c-a826-b9e08a1f5b89', 'name': 'Zürich', 'latitude': 47.378261, 'longitude': 8.542042, 'radius': 10000, 'enabled': True}, {'id': '142129a4-9a4e-432f-aa92-e71b6f4d0518', 'name': 'California Test', 'latitude': 34.05, 'longitude': -118.25, 'radius': 25000, 'enabled': True}, {'id': '557ed145-6d89-4a06-b917-5c358cd0c5a1', 'name': 'Mexico Test', 'latitude': 19.0377, 'longitude': -90.6825, 'radius': 25000, 'enabled': True}, {'id': '2c753721-f9b6-455e-a9d7-73cb350f57d5', 'name': 'Mexico Test', 'latitude': 19.0377, 'longitude': -90.6825, 'radius': 25000, 'enabled': True}]}


In [3]:
payload = {
    "latitude": 19.0377,
    "longitude": -90.6825,
    "name": "Mexico Test",
    "radius": 25000   
}

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.post(url, headers=headers, json=payload)

print(response.status_code)
data = response.json()
print(data)

201
{'data': {'id': '2c753721-f9b6-455e-a9d7-73cb350f57d5', 'name': 'Mexico Test', 'latitude': 19.0377, 'longitude': -90.6825, 'radius': 25000, 'enabled': True}}


In [3]:
import requests

location_id = "557ed145-6d89-4a06-b917-5c358cd0c5a1"

# Append query string directly in URL
url = "https://fireping.net/api/v1/fires/user?hours=168&limit=1000"

headers = {
    "Authorization": f"Bearer {api_key}"
}

response = requests.get(url, headers=headers)

print(response.status_code)
fires = response.json()
print(fires)

200
{'data': [{'latitude': 19.04805, 'longitude': -90.91425, 'detected_at': '2026-05-08T08:22:00Z', 'confidence': 'n', 'frp': 0.84, 'satellite': 'N20'}, {'latitude': 19.01908, 'longitude': -90.60088, 'detected_at': '2026-05-08T08:22:00Z', 'confidence': 'n', 'frp': 2.46, 'satellite': 'N20'}, {'latitude': 19.04934, 'longitude': -90.91478, 'detected_at': '2026-05-08T08:22:00Z', 'confidence': 'n', 'frp': 1.61, 'satellite': 'N20'}, {'latitude': 19.01988, 'longitude': -90.60454, 'detected_at': '2026-05-08T08:22:00Z', 'confidence': 'n', 'frp': 3.13, 'satellite': 'N20'}, {'latitude': 19.05091, 'longitude': -90.9157, 'detected_at': '2026-05-08T08:03:00Z', 'confidence': 'n', 'frp': 1.78, 'satellite': 'N'}, {'latitude': 19.0202, 'longitude': -90.60274, 'detected_at': '2026-05-08T08:03:00Z', 'confidence': 'n', 'frp': 4.02, 'satellite': 'N'}, {'latitude': 19.09306, 'longitude': -90.68861, 'detected_at': '2026-05-08T08:03:00Z', 'confidence': 'n', 'frp': 1.47, 'satellite': 'N'}, {'latitude': 18.82158

In [9]:
import pandas as pd
import folium

df = pd.DataFrame(fires['data'])

confidence_map = {"l": "Low", "n": "Nominal", "h": "High"}

m = folium.Map(location=[19.04805, -90.91425], zoom_start=10)

for _, fire in df.iterrows():
    # Split date and time
    dt = fire['detected_at'].split("T")
    date = dt[0]
    time = dt[1].replace("Z", "")
    
    # Convert confidence
    conf_text = confidence_map.get(fire['confidence'], fire['confidence'])
    
    # Popup text
    popup_text = (
        f"Date: {date}<br>"
        f"Time: {time}<br>"
        f"Confidence: {conf_text}<br>"
        f"FRP: {fire['frp']}<br>"
        f"Satellite: {fire['satellite']}"
    )
    
    folium.CircleMarker(
        location=[fire['latitude'], fire['longitude']],
        radius=6,
        popup=popup_text,
        color='red',
        fill=True,
        fill_opacity=0.7
    ).add_to(m)

radius_m = 25000

folium.Circle(
    location=[19.0377, -90.6825],
    radius=radius_m,       # in meters
    color="blue",
    fill=True,
    fill_opacity=0.1,
    popup="Monitoring Area: 25 km radius"
).add_to(m)

m